In [1]:
import pandas as pd

df = pd.read_csv('/content/superstore.csv')

print(df.shape)
print(df.columns.tolist())

(51290, 27)
['Category', 'City', 'Country', 'Customer.ID', 'Customer.Name', 'Discount', 'Market', '记录数', 'Order.Date', 'Order.ID', 'Order.Priority', 'Product.ID', 'Product.Name', 'Profit', 'Quantity', 'Region', 'Row.ID', 'Sales', 'Segment', 'Ship.Date', 'Ship.Mode', 'Shipping.Cost', 'State', 'Sub.Category', 'Year', 'Market2', 'weeknum']


In [ ]:
import sqlite3
conn=sqlite3.connect('Superstore_Analysis.db')

df.to_sql('superstore',conn, if_exists='replace',index=False)

print("SQLite database created successfully!")

SQLite database created successfully!


In [ ]:
cursor=conn.cursor()
cursor.execute("SELECT COUNT(*) FROM Superstore")

print(cursor.fetchone())

(51290,)


In [ ]:
query="""
SELECT
  COUNT(*) AS total_records,
  COUNT("Order.ID") AS order_id_present,
  COUNT("Order.Date") AS order_date_present,
  COUNT("Customer.ID") AS customer_id_present,
  COUNT("Product.ID") AS product_id_present,
  COUNT("Sales") AS sales_present,
  COUNT("Profit") AS profit_present,
  COUNT("Quantity") AS quantity_present
FROM superstore
"""

result=pd.read_sql_query(query, conn)
result

,total_records,order_id_present,order_date_present,customer_id_present,product_id_present,sales_present,profit_present,quantity_present
0,51290,51290,51290,51290,51290,51290,51290,51290


In [ ]:
query="""
SELECT
  COUNT(*) AS total_records,
  COUNT(DISTINCT "Row.ID") AS unique_row_ids
FROM superstore;
"""

result= pd.read_sql_query(query,conn)
result

,total_records,unique_row_ids
0,51290,51290


In [ ]:
query="""
SELECT
  MIN("Order.Date") AS earliest_order_date,
  MAX("Order.Date") AS latest_order_date,
  MIN("Ship.Date") AS earliest_ship_date,
  MAX("Ship.Date") AS latest_ship_date
FROM superstore;
"""
result=pd.read_sql_query(query,conn)
result

,earliest_order_date,latest_order_date,earliest_ship_date,latest_ship_date
0,2011-01-01 00:00:00.000,2014-12-31 00:00:00.000,2011-01-03 00:00:00.000,2015-01-07 00:00:00.000


In [ ]:
query="""
SELECT
  COUNT (*) AS invalid_shipping_dates
FROM superstore
WHERE "Ship.Date" < "Order.Date";
"""
result=pd.read_sql_query(query,conn)
result


,invalid_shipping_dates
0,0


In [ ]:
query="""
SELECT
  COUNT (*) AS negative_sales_records
FROM superstore
WHERE sales <0;
"""
result=pd.read_sql_query(query,conn)
result

,negative_sales_records
0,0


In [ ]:
query="""
SELECT
  COUNT (*) as zero_sales_records
FROM superstore
WHERE sales= 0;
"""

result= pd.read_sql_query(query, conn)
result

,zero_sales_records
0,1


In [ ]:
query="""
SELECT
  "Order.ID",
  "Order.Date",
  "Customer.ID",
  "Product.ID",
  "Product.Name",
  Quantity,
  Sales,
  Profit
FROM superstore
WHERE sales = 0;
"""

result=pd.read_sql_query(query,conn)
result


,Order.ID,Order.Date,Customer.ID,Product.ID,Product.Name,Quantity,Sales,Profit
0,US-2014-102288,2014-06-20 00:00:00.000,ZC-219104,OFF-AP-10002906,Hoover Replacement Belt for Commercial Guardsm...,1,0,-1.11


In [ ]:
query="""
SELECT
  COUNT(*) AS negative_profit_records
FROM superstore
WHERE Profit <0;
"""
result=pd.read_sql_query(query,conn)
result

,negative_profit_records
0,12544


In [ ]:
query= """
SELECT
  COUNT (*) AS market_mismatch_records
FROM superstore
WHERE Market <> Market2;
"""

result= pd.read_sql_query(query,conn)
result

,market_mismatch_records
0,10378


In [ ]:
query= """
SELECT
  Market,
  Market2,
  COUNT (*) AS record_count
FROM superstore
WHERE Market <> Market2
GROUP BY Market, Market2
ORDER BY record_count DESC;
"""

result= pd.read_sql_query(query,conn)
result

,Market,Market2,record_count
0,US,North America,9994
1,Canada,North America,384


In [ ]:
query="""
SELECT
  SUM(Sales) AS total_sales,
  SUM(Profit) AS total_profit
FROM superstore;
"""

result=pd.read_sql_query(query,conn)
result

,total_sales,total_profit
0,12642905,1.467457e+06


In [ ]:
query="""
SELECT
  SUM(Sales) AS total_sales,
  SUM(Profit) AS total_profit,
  ROUND(SUM(Profit)*100 / SUM(Sales) , 2) AS profit_margin
FROM superstore;
"""

result=pd.read_sql_query(query,conn)
result

,total_sales,total_profit,profit_margin
0,12642905,1.467457e+06,11.61


In [ ]:
query="""
SELECT
  Category,
  SUM(Sales) AS total_sales,
  SUM(Profit) AS total_profit,
  ROUND(SUM(Profit)*100.0/SUM(Sales),2) AS profit_margin
FROM superstore
GROUP BY Category
ORDER BY total_sales DESC;
"""

result= pd.read_sql_query(query, conn)
result

,Category,total_sales,total_profit,profit_margin
0,Technology,4744691,663778.73318,13.99
1,Furniture,4110884,285204.72380,6.94
2,Office Supplies,3787330,518473.83430,13.69


In [ ]:
query="""
SELECT
  Category,
  "Sub.category",
  SUM(Sales) AS total_sales,
  SUM(Profit) AS total_profit,
  ROUND(SUM(Profit)*100 / SUM(Sales),2) AS profit_margin
FROM superstore
GROUP BY Category, "Sub.category"
ORDER BY total_profit ASC;
"""

result= pd.read_sql_query(query, conn)
result

,Category,Sub.Category,total_sales,total_profit,profit_margin
0,Furniture,Tables,757034,-64083.38870,-8.47
1,Office Supplies,Fasteners,83254,11525.42410,13.84
2,Office Supplies,Labels,73433,15010.51200,20.44
3,Office Supplies,Supplies,243090,22583.26310,9.29
4,Office Supplies,Envelopes,170926,29601.11630,17.32
5,Furniture,Furnishings,385609,46967.42550,12.18
6,Office Supplies,Art,372163,57953.91090,15.57
7,Technology,Machines,779071,58867.87300,7.56
8,Office Supplies,Paper,244307,59207.68270,24.23
9,Office Supplies,Binders,461952,72449.84600,15.68


In [ ]:
query="""
SELECT
  "Sub.Category",
  ROUND(AVG(Discount), 3) AS average_discount,
  SUM(Sales) AS total_sales,
  SUM(Profit) AS total_profit
FROM superstore
WHERE "Sub.Category" = 'Tables'
GROUP BY "Sub.Category";
"""

result= pd.read_sql_query(query,conn)
result

,Sub.Category,average_discount,total_sales,total_profit
0,Tables,0.291,757034,-64083.3887


In [ ]:
query="""
SELECT
  Discount,
  COUNT(*) AS order_lines,
  SUM(Sales) AS total_sales,
  SUM(Profit) AS total_profit,
  ROUND(SUM(Profit)*100/SUM(Sales), 2) AS profit_margin_pct
FROM superstore
WHERE "Sub.Category"= 'Tables'
GROUP BY Discount
ORDER BY Discount
"""

result=pd.read_sql_query(query,conn)
result

,Discount,order_lines,total_sales,total_profit,profit_margin_pct
0,0.00,208,261807,60447.1997,23.09
1,0.10,3,3481,795.0270,22.84
2,0.20,183,157721,7215.5360,4.57
3,0.30,116,104747,-11960.4786,-11.42
4,0.35,45,54159,-9122.6490,-16.84
5,0.40,85,52162,-17105.1808,-32.79
6,0.45,13,8032,-3596.3026,-44.77
7,0.47,34,32502,-10680.2751,-32.86
8,0.50,52,26505,-15049.8623,-56.78
9,0.55,10,6308,-3150.6720,-49.95


In [ ]:
query="""
SELECT
  "Sub.Category",
  ROUND(AVG("Shipping.Cost"), 2) AS avg_shipping_cost,
  SUM(Sales) AS total_sales,
  SUM(Profit) AS total_profit
FROM superstore
WHERE "Sub.Category"= 'Tables'
GROUP BY "Sub.Category";
"""
result=pd.read_sql_query(query,conn)
result


,Sub.Category,avg_shipping_cost,total_sales,total_profit
0,Tables,92.75,757034,-64083.3887


In [ ]:
query="""
SELECT
  CASE
    WHEN "Shipping.Cost" < 25 THEN 'Under $25'
    WHEN "Shipping.Cost" < 50 THEN '$25-$49'
    WHEN "Shipping.Cost" < 100 THEN '$50-$99'
    WHEN "Shipping.Cost" < 200 THEN '$100-$199'
    ELSE '$200+'
  END AS shipping_cost_range,
  COUNT(*) AS order_lines,
  SUM(Profit) AS total_profit,
  SUM(Sales) AS total_sales,
  ROUND(SUM(Profit)*100.0 / SUM(Sales), 2) AS proft_margin_pct
FROM superstore
WHERE "Sub.Category"= 'Tables'
GROUP BY shipping_cost_range
ORDER BY
  CASE shipping_cost_range
    WHEN 'Under $25' THEN 1
    WHEN '$25-$49' THEN 2
    WHEN '$50-$99' THEN 3
    WHEN '$100-$199' THEN 4
    WHEN '$200+' THEN 5
  END;
"""

result=pd.read_sql_query(query,conn)
result


,shipping_cost_range,order_lines,total_profit,total_sales,proft_margin_pct
0,Under $25,191,-24439.3740,56026,-43.62
1,$25-$49,200,-20733.2149,98622,-21.02
2,$50-$99,226,-25356.8215,192981,-13.14
3,$100-$199,140,-7044.3827,188200,-3.74
4,$200+,104,13490.4044,221205,6.10


In [ ]:
query="""
SELECT
  "Product.ID",
  "Product.Name",
  COUNT(*) AS order_lines,
  SUM(Sales) AS total_sales,
  SUM(Profit) AS total_profit,
  ROUND(SUM(Profit) * 100.0/ SUM(Sales), 2) AS profit_margin_pct,
  ROUND(AVG(DISCOUNT)*100, 2) AS avg_discount_pct
FROM superstore
WHERE "Sub.Category"= 'Tables'
GROUP BY "Product.ID", "Product.Name"
ORDER BY total_profit ASC
LIMIT 15
"""

result=pd.read_sql_query(query,conn)
result


,Product.ID,Product.Name,order_lines,total_sales,total_profit,profit_margin_pct,avg_discount_pct
0,FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tabl...,5,9918,-2876.1156,-29.00,28.00
1,FUR-TA-10002885,"Lesro Computer Table, Fully Assembled",3,1200,-2798.4880,-233.21,70.00
2,FUR-TA-10002172,"Hon Conference Table, Rectangular",3,3667,-2619.3105,-71.43,59.00
3,FUR-BAR-10003532,"Barricks Conference Table, Rectangular",2,3076,-2460.8100,-80.00,35.00
4,FUR-TA-10001633,"Bevis Wood Table, with Bottom Storage",4,6724,-2113.6140,-31.43,30.00
5,FUR-TA-10003354,"Barricks Conference Table, Adjustable Height",2,3728,-2082.3930,-55.86,52.50
6,FUR-TA-10002972,"Barricks Computer Table, Fully Assembled",3,3633,-1965.5502,-54.10,50.33
7,FUR-TA-10001889,Bush Advantage Collection Racetrack Conference...,7,9546,-1934.3976,-20.26,35.00
8,FUR-TA-10003963,"Bevis Conference Table, Fully Assembled",1,555,-1924.5420,-346.76,85.00
9,FUR-TA-10004054,"Lesro Conference Table, with Bottom Storage",2,2830,-1852.5660,-65.46,47.50


In [ ]:
query = """
SELECT
    profitability_status,
    COUNT(*) AS product_count,
    ROUND(SUM(total_sales), 2) AS total_sales,
    ROUND(SUM(total_profit), 2) AS total_profit
FROM (
    SELECT
        "Product.ID",
        CASE
            WHEN SUM(Profit) < 0 THEN 'Unprofitable'
            WHEN SUM(Profit) > 0 THEN 'Profitable'
            ELSE 'Break-even'
        END AS profitability_status,
        SUM(Sales) AS total_sales,
        SUM(Profit) AS total_profit
    FROM superstore
    WHERE "Sub.Category" = 'Tables'
    GROUP BY "Product.ID"
)
GROUP BY profitability_status
ORDER BY total_profit ASC;
"""

result = pd.read_sql_query(query, conn)
result

,profitability_status,product_count,total_sales,total_profit
0,Unprofitable,210,430049.0,-117166.37
1,Break-even,1,2720.0,0.00
2,Profitable,155,324265.0,53082.98


In [ ]:
query="""
WITH product_summary AS (
  SELECT
    "Product.ID",
    SUM(Profit) AS total_profit,
    AVG(Discount) * 100 AS avg_discount_pct
  FROM superstore
  WHERE "Sub.Category" = 'Tables'
  GROUP BY "Product.ID"
)
SELECT
  CASE
    WHEN total_profit <0 THEN 'Unprofitable'
    WHEN total_profit >0 THEN 'Profitable'
    ELSE 'Break-even'
  END AS profitability_status,
  COUNT(*) AS product_count,
  ROUND(AVG(avg_discount_pct), 2) AS avg_discount_pct,
  ROUND(SUM(total_profit), 2) AS total_profit
FROM product_summary
GROUP BY profitability_status
ORDER BY total_profit ASC;
"""

result=pd.read_sql_query(query,conn)
result

,profitability_status,product_count,avg_discount_pct,total_profit
0,Unprofitable,210,40.15,-117166.37
1,Break-even,1,40.00,0.00
2,Profitable,155,15.01,53082.98


In [ ]:
# Product with highest total sales
product_sales=(
    df.groupby(["Product.ID", "Product.Name"], as_index=False)
    .agg(
        Total_Sales=("Sales", "sum"),
        Order_Lines=("Order.ID", "count")
    )
    .sort_values("Total_Sales", ascending=False)
)
print("TOP 10 PRODUCTS BY TOTAL SALES")
display(product_sales.head(10).round(2))

# Product with highest total profit
product_profit=(
    df.groupby(["Product.ID", "Product.Name"], as_index=False)
    .agg(
        Total_Profit=("Profit", "sum"),
        Order_Lines=("Order.ID", "count")
    )
    .sort_values("Total_Profit", ascending=False)
)
print("TOP 10 PRODUCTS BY TOTAL PROFIT")
display(product_profit.head(10).round(2))


TOP 10 PRODUCTS BY TOTAL SALES


,Product.ID,Product.Name,Total_Sales,Order_Lines
9388,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61600,5
10632,TEC-PH-10004664,"Nokia Smart Phone, with Caller ID",30042,10
4017,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,27454,10
9721,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,22638,1
10646,TEC-PH-10004823,"Nokia Smart Phone, Full Size",22261,11
745,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,21870,8
541,FUR-CH-10000027,"SAFCO Executive Leather Armchair, Black",21329,12
2882,OFF-AP-10004512,"Hoover Stove, Red",21148,7
3744,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,19824,11
3634,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,19026,9


TOP 10 PRODUCTS BY TOTAL PROFIT


,Product.ID,Product.Name,Total_Profit,Order_Lines
9388,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,25199.93,5
2882,OFF-AP-10004512,"Hoover Stove, Red",10345.58,7
10646,TEC-PH-10004823,"Nokia Smart Phone, Full Size",8121.48,11
4017,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,7753.04,10
9099,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,6983.88,8
776,FUR-CH-10002250,"Office Star Executive Leather Armchair, Black",6123.26,10
10632,TEC-PH-10004664,"Nokia Smart Phone, with Caller ID",5455.95,10
2681,OFF-AP-10002330,"Hamilton Beach Stove, Silver",5452.46,7
10097,TEC-PH-10000303,"Samsung Smart Phone, VoIP",5356.81,6
766,FUR-CH-10002203,"SAFCO Executive Leather Armchair, Black",5003.10,5


In [ ]:
# Analyze profitability by customer segment
segment_analysis=(
    df.groupby("Segment", as_index=False)
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Number_of_Orders=("Order.ID", "nunique"),
        Number_of_Customers=("Customer.ID", "nunique")
    )
)

#Calculate profit margin as percentage of sales
segment_analysis["Profit_Margin_Percentage"]=(
    segment_analysis["Total_Profit"]
    / segment_analysis["Total_Sales"]
    *100
)

# Rank segment by total profit, highest first
segment_analysis=segment_analysis.sort_values(
    "Total_Profit", ascending=False
)

display(segment_analysis.round(2))

,Segment,Total_Sales,Total_Profit,Number_of_Orders,Number_of_Customers,Profit_Margin_Percentage
0,Consumer,6508141,749239.78,13104,2509,11.51
1,Corporate,3824808,441208.33,7673,1457,11.54
2,Home Office,2309956,277009.18,4687,907,11.99


In [ ]:
# Find 10 products with the lowest total profit

worst_performing_products=(
    df.groupby(["Product.ID", "Product.Name"], as_index=False)
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Order_Lines=("Order.ID", "count")
    )
    .sort_values("Total_Profit", ascending=True)
    .head(10)
)

#Calculate aggegrate Profit Margin

worst_performing_products["Profit_Margin_Percentage"]= (
    worst_performing_products["Total_Profit"]
    / worst_performing_products["Total_Sales"]
    *100
)

display(worst_performing_products.round(2))

,Product.ID,Product.Name,Total_Sales,Total_Profit,Order_Lines,Profit_Margin_Percentage
9574,TEC-MA-10000418,Cubify CubeX 3D Printer Double Head Print,11100,-8879.97,3,-80.00
2614,OFF-AP-10001623,"Hoover Stove, White",11730,-4958.16,6,-42.27
9605,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,16830,-4589.97,4,-27.27
10424,TEC-PH-10002991,"Apple Smart Phone, Full Size",7258,-4574.64,4,-63.03
9984,TEC-MOT-10003050,"Motorola Smart Phone, Cordless",10350,-3998.68,7,-38.63
9876,TEC-MA-10004125,Cubify CubeX 3D Printer Triple Head Print,8000,-3839.99,1,-48.00
699,FUR-CH-10001582,"Office Star Executive Leather Armchair, Black",6498,-3066.78,5,-47.20
1993,FUR-TA-10000198,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9918,-2876.12,5,-29.00
2157,FUR-TA-10002885,"Lesro Computer Table, Fully Assembled",1200,-2798.49,3,-233.21
2118,FUR-TA-10002172,"Hon Conference Table, Rectangular",3667,-2619.31,3,-71.43


In [ ]:
# Analyze profitability across shipping-cost bands

shipping_analysis=df.copy()
shipping_analysis["Shipping_Cost_Band"]= pd.cut(
    shipping_analysis["Shipping.Cost"],
    bins=[-float("inf"), 25, 50, 100, 200, float("inf")],
    labels=[
        "Under $25",
        "$25-$49",
        "$50-$99",
        "$100-$199",
        "$200+"
    ],
right=False
)

shipping_summary=(
    shipping_analysis.groupby("Shipping_Cost_Band", observed=False)
.agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Order_Lines=("Order.ID", "count")
)
.reset_index()
)


shipping_summary["Profit_Margin_Percentage"]= (
    shipping_summary["Total_Profit"]
    / shipping_summary["Total_Sales"] *100
)

display(shipping_summary.round(2))


,Shipping_Cost_Band,Total_Sales,Total_Profit,Order_Lines,Profit_Margin_Percentage
0,Under $25,3457151,269849.64,38702,7.81
1,$25-$49,2162070,224113.96,5743,10.37
2,$50-$99,2592291,302054.76,3858,11.65
3,$100-$199,2226086,295242.80,1960,13.26
4,$200+,2205307,376196.12,1027,17.06


In [ ]:
# Analyzing association between discount and profitability across whole dataset

discount_analysis=(
    df.groupby("Discount", as_index=False)
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Order_Lines=("Order.ID", "count")
    )
)

discount_analysis["Profit_Margin_Percentage"]=(
    discount_analysis["Total_Profit"]
    / discount_analysis["Total_Sales"] *100
)

display(discount_analysis.round({
    "Discount": 3,
    "Total_Sales": 2,
    "Total_Profit": 2,
    "Profit_Margin_Percentage": 2
}))

,Discount,Total_Sales,Total_Profit,Order_Lines,Profit_Margin_Percentage
0,0.000,6992734,1770695.27,29009,25.32
1,0.002,261395,57976.58,461,22.18
2,0.070,121620,21148.50,150,17.39
3,0.100,1579618,259064.18,4068,16.40
4,0.150,306037,27375.90,541,8.95
5,0.170,247404,28163.07,735,11.38
6,0.200,1203855,117715.87,4998,9.78
7,0.202,16213,-595.27,41,-3.67
8,0.250,88063,800.59,198,0.91
9,0.270,89351,-1675.08,388,-1.87


In [ ]:
print("Discount data type:", df["Discount"].dtype)

print("\nExact distinct discount values:")
for value in sorted(df["Discount"].dropna().unique()):
    print(repr(value))

Discount data type: float64

Exact distinct discount values:
np.float64(0.0)
np.float64(0.002)
np.float64(0.07)
np.float64(0.1)
np.float64(0.15)
np.float64(0.17)
np.float64(0.2)
np.float64(0.202)
np.float64(0.25)
np.float64(0.27)
np.float64(0.3)
np.float64(0.32)
np.float64(0.35)
np.float64(0.37)
np.float64(0.4)
np.float64(0.402)
np.float64(0.45)
np.float64(0.47)
np.float64(0.5)
np.float64(0.55)
np.float64(0.57)
np.float64(0.6)
np.float64(0.602)
np.float64(0.65)
np.float64(0.7)
np.float64(0.8)
np.float64(0.85)


In [ ]:
# Group records by exact discount level
discount_analysis= df.copy()

discount_analysis["Discount_Level"]= (discount_analysis["Discount"])

discount_summary=(
    discount_analysis.groupby("Discount_Level", as_index=False)
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Order_Lines=("Order.ID", "count")
    )
)

discount_summary["Profit_Margin_Percentage"]=(
    discount_summary["Total_Profit"] / discount_summary["Total_Sales"] *100
)

display(discount_summary.round({
    "Discount_Level": 3,
    "Total_Sales": 2,
    "Total_Profit": 2,
    "Profit_Margin_Percentage": 2
}))




,Discount_Level,Total_Sales,Total_Profit,Order_Lines,Profit_Margin_Percentage
0,0.000,6992734,1770695.27,29009,25.32
1,0.002,261395,57976.58,461,22.18
2,0.070,121620,21148.50,150,17.39
3,0.100,1579618,259064.18,4068,16.40
4,0.150,306037,27375.90,541,8.95
5,0.170,247404,28163.07,735,11.38
6,0.200,1203855,117715.87,4998,9.78
7,0.202,16213,-595.27,41,-3.67
8,0.250,88063,800.59,198,0.91
9,0.270,89351,-1675.08,388,-1.87


In [ ]:
# Finding Average Order Value (AOV)

total_sales=df["Sales"].sum()
total_order=df["Order.ID"].nunique()

average_order_value= total_sales/total_order

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Orders: {total_order:,}")
print(f"Average Order Value: ${average_order_value:,.2f}")


Total Sales: $12,642,905.00
Total Orders: 25,035
Average Order Value: $505.01


In [ ]:
# Finding Average Shipping Time

order_date=pd.to_datetime(df["Order.Date"])
ship_date=pd.to_datetime(df["Ship.Date"])

shipping_days= (ship_date - order_date).dt.days

average_shipping_time= shipping_days.mean()

print(f"Average Shipping Time: {average_shipping_time:.2f}days")

Average Shipping Time: 3.97days


In [2]:
# Number of Customers

total_customer=df["Customer.ID"].nunique()

print(f"Number Of Customers: {total_customer:,}")

Number Of Customers: 4,873
